# Phase 1: Exploratory Data Analysis of MILDSum Legal Dataset

**Project:** Detecting Hallucinations in AI-Generated Text using NLI and Self-Consistency Checking  
**Context:** Indian court judgments and expert-written English summaries from the **MILDSum** benchmark.  

In this notebook, we:
1. Load the processed dataset (`sample_pairs.jsonl` and `stats.csv`).
2. Inspect 3 real judgment-summary pairs side by side.
3. Analyze length and sentence distributions.
4. Highlight critical practical challenges for NLI (SummaC) and LLM consistency checking (SelfCheckGPT).

In [ ]:
import json
import os
import pandas as pd

processed_dir = os.path.join("..", "data", "processed")
stats_path = os.path.join(processed_dir, "stats.csv")
jsonl_path = os.path.join(processed_dir, "sample_pairs.jsonl")

df_stats = pd.read_csv(stats_path)
print(f"Loaded stats for {len(df_stats)} document pairs.")
df_stats.head(10)

## 1. Length Distributions and Statistics

In [ ]:
metrics = [
    "judgment_chars", "judgment_words", "judgment_sentences",
    "summary_chars", "summary_words", "summary_sentences",
    "compression_ratio_words", "compression_ratio_sents"
]

summary_table = df_stats[metrics].describe().T[["mean", "std", "min", "50%", "max"]]
summary_table.rename(columns={"50%": "median"}, inplace=True)
print("=== LENGTH DISTRIBUTION SUMMARY ===")
summary_table

## 2. Side-by-Side Inspection of 3 Example Pairs

In [ ]:
pairs = []
with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        pairs.append(json.loads(line))

for item in pairs[:3]:
    pid = item["id"]
    j_text = item["judgment_text"]
    s_text = item["summary_text"]
    j_sents = item["judgment_sentences"]
    s_sents = item["summary_sentences"]
    
    print("=" * 80)
    print(f"CASE ID: {pid}")
    print(f"Judgment: {len(j_text.split())} words, {len(j_sents)} sentences")
    print(f"Summary:  {len(s_text.split())} words, {len(s_sents)} sentences")
    print("-" * 80)
    print("JUDGMENT (First 2 sentences):")
    for s in j_sents[:2]:
        print(f"  > {s}")
    print("\nSUMMARY (First 3 sentences):")
    for s in s_sents[:3]:
        print(f"  * {s}")
    print("=" * 80 + "\n")

## 3. Key Observations & Potential Trouble Points for Downstream Phases

From our Phase 1 exploratory data analysis, we identify 4 key observations that must guide Phase 2 (Evaluation) and Phase 3/4 (Approach A & B):

1. **Severe Document Length vs. Model Input Limit (SummaC / NLI):**
   - Average judgment length is **~3,078 words (approx. 4,000+ tokens)**, with the longest judgment reaching **5,959 words (~8,000 tokens)**.
   - Standard NLI models (RoBERTa / DeBERTa used by SummaC) have a maximum context window of **512 tokens**.
   - *Implication:* We cannot pass the entire judgment into the NLI model at once. In Approach A, we must either chunk the judgment into paragraphs or use sentence retrieval / top-K paragraph matching before evaluating NLI entailment for each summary sentence.

2. **Indian Legal Citations and Currency Formatting:**
   - Summary sentences frequently contain abbreviations (`Rs. 24,02,380`, `GNCTD`, `ADJ Dinesh Kumar`, `Ors. vs. State`, `Sec. 302 IPC`).
   - Standard off-the-shelf splitters break sentences inside amounts like `Rs. 5,000`. Our custom NLTK Legal Punkt tokenizer successfully prevents these false splits.

3. **Compression Ratio Variance:**
   - Word compression ratios range from **12.5% to 59.6%** (average ~29.8%).
   - Summaries are dense and factual: they contain exact figures, hospital names, dates, sections, and court orders.
   - Hallucination detection must focus precisely on these factual claims (names, amounts, and rulings).

4. **Dataset Availability Note:**
   - The official MILDSum GitHub repository hosts 10 sample pairs publicly in `Data/MILDSum_Samples/`.
   - For the full 3,122 cases, researchers contact the authors (`debtanudatta04@gmail.com`).
   - Our pipeline seamlessly accommodates any number of added sample folders in `data/raw/`.